# 12. Historical Value at Risk Engine

## Objective

This notebook demonstrates how historical market scenarios can be replayed to generate a distribution of profit and loss (P&L).

Workflow

Historical Scenarios
→ Shock Today's Yield Curve
→ Reprice Interest Rate Swap
→ Compute Scenario P&L

At this stage, we are generating the historical P&L distribution. In the next module, we will calculate Historical VaR statistics such as the 95% and 99% VaR.

In [1]:
from datetime import date

from src.common.enums import (
    DayCountConvention,
    FloatingIndex,
    PayReceive,
    PaymentFrequency,
)

from src.curves.bootstrapper import Bootstrapper

from src.market_data.market_instrument import MarketInstrument
from src.market_data.market_quote_collection import MarketQuoteCollection

from src.products.trade import Trade
from src.products.fixed_leg import FixedLeg
from src.products.floating_leg import FloatingLeg
from src.products.interest_rate_swap import InterestRateSwap

from src.risk.historical_scenario import HistoricalScenario
from src.risk.historical_var_engine import HistoricalVaREngine

## Build Yield Curve

In [2]:
quotes = MarketQuoteCollection()

quotes.add(MarketInstrument("Deposit", "1M", 0.05))
quotes.add(MarketInstrument("Deposit", "3M", 0.05))
quotes.add(MarketInstrument("Deposit", "6M", 0.05))
quotes.add(MarketInstrument("Deposit", "1Y", 0.05))
quotes.add(MarketInstrument("Deposit", "2Y", 0.05))
quotes.add(MarketInstrument("Deposit", "3Y", 0.05))

curve = Bootstrapper(
    valuation_date=date(2026, 1, 1),
    market_quotes=quotes,
).build()

curve

YieldCurve(valuation_date=2026-01-01, points=6)

## Build Interest Rate Swap

In [3]:
trade = Trade(
    trade_id="IRS001",
    counterparty="ABC",
    currency="USD",
    effective_date=date(2026, 1, 1),
    maturity_date=date(2029, 1, 1),
)

fixed_leg = FixedLeg(
    notional=1_000_000,
    fixed_rate=0.05,
    payment_frequency=PaymentFrequency.ANNUAL,
    day_count=DayCountConvention.ACT_365,
    pay_receive=PayReceive.RECEIVE,
)

floating_leg = FloatingLeg(
    notional=1_000_000,
    payment_frequency=PaymentFrequency.ANNUAL,
    day_count=DayCountConvention.ACT_365,
    pay_receive=PayReceive.PAY,
    floating_index=FloatingIndex.SOFR,
)

swap = InterestRateSwap(
    trade=trade,
    fixed_leg=fixed_leg,
    floating_leg=floating_leg,
)

swap

InterestRateSwap(trade=Trade(trade_id='IRS001', counterparty='ABC', currency='USD', effective_date=datetime.date(2026, 1, 1), maturity_date=datetime.date(2029, 1, 1)), fixed_leg=FixedLeg(notional=1000000, payment_frequency=<PaymentFrequency.ANNUAL: 1>, day_count=<DayCountConvention.ACT_365: 'ACT/365'>, pay_receive=<PayReceive.RECEIVE: 'RECEIVE'>, fixed_rate=0.05), floating_leg=FloatingLeg(notional=1000000, payment_frequency=<PaymentFrequency.ANNUAL: 1>, day_count=<DayCountConvention.ACT_365: 'ACT/365'>, pay_receive=<PayReceive.PAY: 'PAY'>, floating_index=<FloatingIndex.SOFR: 'SOFR'>, spread=0.0))

## Create Historical Scenarios

Each historical scenario represents a historical movement in the yield curve. Different tenors may move by different amounts.

In [4]:
historical_scenarios = [

    HistoricalScenario(
        scenario_date=date(2025, 1, 14),
        tenor_shifts={
            "1Y": 0.0003,
            "2Y": 0.0005,
            "3Y": 0.0007,
        },
    ),

    HistoricalScenario(
        scenario_date=date(2025, 1, 15),
        tenor_shifts={
            "1Y": -0.0002,
            "2Y": -0.0004,
            "3Y": -0.0005,
        },
    ),

    HistoricalScenario(
        scenario_date=date(2025, 1, 16),
        tenor_shifts={
            "1Y": 0.0008,
            "2Y": 0.0006,
            "3Y": 0.0002,
        },
    ),
]

## Replay Historical Scenarios

In [5]:
results = HistoricalVaREngine().run(
    swap,
    curve,
    historical_scenarios,
)

In [6]:
print(
    f"{'Scenario Date':<15}"
    f"{'Present Value':>20}"
    f"{'PnL':>15}"
)

print("-" * 55)

for result in results:

    print(
        f"{result.scenario:<15}"
        f"{result.present_value:>20,.2f}"
        f"{result.pnl:>15,.2f}"
    )

Scenario Date         Present Value            PnL
-------------------------------------------------------
2025-01-14               135,806.63        -150.01
2025-01-15               136,067.15         110.50
2025-01-16               135,838.47        -118.18


## Interpretation

Each row represents the portfolio valuation under a historical market move.

The engine:

1. Applies the historical tenor shifts to today's yield curve.
2. Reprices the swap using the shocked curve.
3. Calculates the resulting profit or loss.

The collection of P&L values forms the historical P&L distribution. In the next module, this distribution will be analyzed to calculate Historical Value at Risk (VaR).